# Deep Live Cam - Colab Live Streaming (Debug Edition)

Notebook ini **khusus Live Streaming** dan sudah dilengkapi logging + auto-diagnosa.
Server di sini memperbaiki bug notebook lama: `pull_worker` lama `break` setelah satu upload,
sehingga Live kedua dan seterusnya selalu timeout di `Out S:`.

## Aturan alamat (WAJIB, dari printout cell tunnel)
| GUI field | Tunnel notebook | Port server | Peran |
|---|---|---|---|
| `In:` | `[output]` | 5557 | server mengirim frame hasil ke GUI (ffmpeg mpegts, BUKAN ZMQ) |
| `Out S:` | `[input]` | 5555 | GUI mengirim source face (ZMQ REQ -> REP) |
| `Out T:` | `[inputtwo]` | 5556 | GUI mengirim frame webcam (ffmpeg mpegts, BUKAN ZMQ) |

## Urutan menjalankan
1. Cell **1 Install** -> **2 Model** -> **3 Logger** -> **4 Face engine** -> **5 Server** -> **6 Tunnel**
2. Copy 3 URL dari cell tunnel ke GUI sesuai tabel di atas.
3. **7 Diagnosa** (opsional, sangat disarankan) untuk cek port + tunnel + uji ZMQ lokal.
4. Di GUI: pilih Source, klik **Live sekali**. Tidak perlu re-run cell server untuk Live berikutnya.

## Cara baca log
Semua baris berformat `HH:MM:SS [TAG] pesan` dan juga ditulis ke `/content/live_debug.log`:

| Tag | Arti |
|---|---|
| `BOOT` | start/stop thread server |
| `SRC` | upload source face dari GUI (metadata, chunk, ACK) |
| `TEM` | decoder frame webcam dari GUI (port 5556) |
| `OUT` | encoder frame hasil ke GUI (port 5557) |
| `SWAP` | proses face swap (error inference muncul di sini) |
| `STAT` | snapshot status tiap 5 detik |
| `HINT` | saran perbaikan otomatis saat ada yang macet |
| `TEST` | hasil self-test ZMQ dari cell diagnosa |
| `TUN` | status tunnel ngrok |


## 1. Install dependency

Pasang onnxruntime-gpu, insightface, pyzmq, dan ffmpeg. Jalankan sekali per sesi Colab.


In [1]:
!pip install -q insightface pyzmq msgpack
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu>=1.20,<1.23" || pip install -q onnxruntime-gpu
!apt-get install -y -q ffmpeg
import subprocess
print(subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True).stdout.splitlines()[0])
import onnxruntime
print('ONNX_PROVIDERS', onnxruntime.get_available_providers())
print('GPU_DEVICE', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'tidak ada GPU terpasang')


Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
ONNX_PROVIDERS ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
GPU_DEVICE Tesla T4, 15360 MiB, 0 MiB


## 2. Model face swap

Cell ini mengunduh `inswapper_128.onnx` (528 MB, dipakai sebagai cadangan) dan model yang dipilih pada `SWAP_MODEL`.

Default `hyperswap_1a_256.onnx` berukuran sekitar 403 MB dan merupakan default resmi FaceFusion sejak versi 3.3.2.
Varian lain yang tersedia: `hyperswap_1b_256`, `hyperswap_1c_256`. Ketiganya memakai template `arcface_128`, input 256x256, mean dan standar deviasi 0.5.

`-nc` membuat unduhan tidak diulang bila file sudah ada. Tiap model dicetak `exists` beserta `size`, jadi unduhan yang gagal langsung terlihat.


In [2]:
import os

MODEL_DIR = '/content/deepfakecollab/Model'
SWAP_MODEL = 'hyperswap_1b_256'
MODEL_SOURCES = {
    'inswapper_128': 'https://huggingface.co/ninjawick/webui-faceswap-unlocked/resolve/main/inswapper_128.onnx',
    'hyperswap_1a_256': 'https://huggingface.co/facefusion/models-3.3.0/resolve/main/hyperswap_1a_256.onnx',
    'hyperswap_1b_256': 'https://huggingface.co/facefusion/models-3.3.0/resolve/main/hyperswap_1b_256.onnx',
    'hyperswap_1c_256': 'https://huggingface.co/facefusion/models-3.3.0/resolve/main/hyperswap_1c_256.onnx',
}
MODEL_FILES = {name: MODEL_DIR + '/' + name + '.onnx' for name in MODEL_SOURCES}
MODEL_PATH = MODEL_FILES[SWAP_MODEL]

!mkdir -p {MODEL_DIR}
!wget -q -nc {MODEL_SOURCES['inswapper_128']} -P {MODEL_DIR}
!wget -q -nc {MODEL_SOURCES[SWAP_MODEL]} -P {MODEL_DIR}

for name, path in MODEL_FILES.items():
    print('MODEL', name, 'exists=' + str(os.path.exists(path)), 'size=' + str(os.path.getsize(path) if os.path.exists(path) else 0))
print('SWAP_MODEL', SWAP_MODEL)


MODEL inswapper_128 exists=True size=554253681
MODEL hyperswap_1a_256 exists=False size=0
MODEL hyperswap_1b_256 exists=True size=402742682
MODEL hyperswap_1c_256 exists=False size=0
SWAP_MODEL hyperswap_1b_256


## 3. Logger, state, dan helper diagnosa

Semua tag log, konstanta port, dan fungsi diagnosa didefinisikan di sini. Wajib dijalankan sebelum cell lain.


In [3]:
import json
import os
import socket
import subprocess
import threading
import time
import traceback
from collections import deque
from datetime import datetime

import cv2
import numpy as np
import zmq

LOG_PATH = '/content/live_debug.log'
PORT_SOURCE = 5555
PORT_TEMP = 5556
PORT_OUT = 5557
STREAM_WIDTH = 960
STREAM_HEIGHT = 540
STREAM_CHANNELS = 3
FRAME_RATE = 60
GOP = '60'
BITRATE = '3M'
MAXRATE = '3M'
BUFSIZE = '5M'
STATUS_INTERVAL_S = 5
TEMP_STALE_S = 10
SOURCE_STALE_S = 90
META_TIMEOUT_MS = 1000
CHUNK_TIMEOUT_MS = 12000

STATE = {
    'source_frame': None,
    'source_face': None,
    'source_face_checked': False,
    'source_shape': None,
    'many_faces': False,
    'source_uploads': 0,
    'source_last_at': 0.0,
    'temp_frames': 0,
    'temp_last_at': 0.0,
    'out_frames': 0,
    'out_last_at': 0.0,
    'errors': 0,
    'stop': threading.Event(),
    'threads': {},
    'tunnels': {},
}
TEMP_BUFFER = deque(maxlen=4)
LOG_LOCK = threading.Lock()


def log(tag, message):
    """Tulis satu baris diagnosa ke console Colab dan ke file log."""
    line = datetime.now().strftime('%H:%M:%S') + ' [' + tag + '] ' + message
    with LOG_LOCK:
        print(line, flush=True)
        with open(LOG_PATH, 'a', encoding='utf-8') as handle:
            handle.write(line + '\n')


def reset_log():
    """Kosongkan file log sebelum sesi baru."""
    with open(LOG_PATH, 'w', encoding='utf-8') as handle:
        handle.write('')
    log('BOOT', 'log file reset')


def age(value):
    """Kembalikan umur data dalam detik, atau None bila belum pernah ada."""
    if not value:
        return None
    return round(time.time() - value, 1)


def status_line():
    """Susun satu baris ringkas berisi kondisi semua jalur data."""
    return (
        'source_uploads=' + str(STATE['source_uploads'])
        + ' source_ready=' + str(STATE['source_shape'] is not None)
        + ' source_idle=' + str(age(STATE['source_last_at']))
        + ' temp_frames=' + str(STATE['temp_frames'])
        + ' temp_idle=' + str(age(STATE['temp_last_at']))
        + ' out_frames=' + str(STATE['out_frames'])
        + ' out_idle=' + str(age(STATE['out_last_at']))
        + ' errors=' + str(STATE['errors'])
        + ' buffer=' + str(len(TEMP_BUFFER))
    )


def hints():
    """Hasilkan saran perbaikan otomatis dari kondisi jalur data."""
    messages = []
    source_idle = age(STATE['source_last_at'])
    temp_idle = age(STATE['temp_last_at'])
    if STATE['source_shape'] is None and (source_idle is None or source_idle > SOURCE_STALE_S):
        messages.append('belum ada upload source: cek GUI Out S -> [input] 5555 lalu klik Live')
    if STATE['source_shape'] is not None and temp_idle is not None and temp_idle > TEMP_STALE_S:
        messages.append('frame webcam berhenti: cek GUI Out T -> [inputtwo] 5556 dan status kamera')
    if STATE['temp_frames'] and STATE['out_frames'] == 0:
        messages.append('encoder belum mengirim frame: cek GUI In -> [output] 5557')
    if STATE['errors']:
        messages.append('tercatat ' + str(STATE['errors']) + ' error, baca baris [SRC]/[TEM]/[OUT]/[SWAP]')
    if STATE['errors'] and STATE['source_uploads'] == 0:
        messages.append('upload source putus: coba REGION = ap di cell tunnel agar latensi jauh lebih kecil dari eu')
    return messages


def port_users(ports):
    """Cari PID yang sedang memakai port tertentu."""
    found = {}
    for port in ports:
        output = subprocess.run(['lsof', '-t', '-i:' + str(port)], capture_output=True, text=True).stdout
        found[port] = [pid for pid in output.splitlines() if pid]
    return found


def kill_port_users(ports):
    """Bebaskan port dari proses lama agar bind atau ffmpeg tidak gagal."""
    for port, pids in port_users(ports).items():
        for pid in pids:
            subprocess.run(['kill', '-9', pid], capture_output=True)
            log('BOOT', 'port ' + str(port) + ' dibebaskan dari PID ' + pid)


def print_environment():
    """Cetak versi library dan status GPU agar masalah dependency langsung terlihat."""
    import onnxruntime
    log('ENV', 'python ' + '.'.join(map(str, os.sys.version_info[:3])))
    log('ENV', 'numpy ' + np.__version__ + ' opencv ' + cv2.__version__ + ' pyzmq ' + zmq.__version__)
    log('ENV', 'onnxruntime providers ' + str(onnxruntime.get_available_providers()))
    ffmpeg = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True).stdout.splitlines()
    log('ENV', ffmpeg[0] if ffmpeg else 'ffmpeg tidak ditemukan')
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    log('ENV', 'gpu ' + (gpu.stdout.strip() or gpu.stderr.strip() or 'tidak tersedia'))


def print_ports():
    """Cetak pemakai tiap port utama supaya konflik proses kelihatan."""
    for port, pids in port_users([PORT_SOURCE, PORT_TEMP, PORT_OUT]).items():
        log('PORT', str(port) + ' users=' + (','.join(pids) if pids else 'kosong'))


## 4. Face engine (HyperSwap 256)

Cell ini memuat detektor wajah, model pengenal wajah untuk source, dan model swap yang dipilih di cell model.

`SWAP_MODEL = 'hyperswap_1a_256'` menjalankan HyperSwap dengan template `arcface_128` dan `PIXEL_BOOST = 256`, artinya crop 256x256 sekali jalan tanpa tiling.
Kontrak model mengikuti FaceFusion: embedding source dinormalisasi L2, crop BGR diubah ke RGB rentang -1 sampai 1, keluaran dikembalikan ke rentang 0 sampai 1 lalu ditempel dengan mask kotak bertepi lembut.

Untuk A/B: ubah `SWAP_MODEL` ke `hyperswap_1b_256`, `hyperswap_1c_256`, atau `inswapper_128`, lalu jalankan ulang cell model, cell logger, dan cell ini.
Baris `warmup` di akhir cell mencetak ms per inferensi plus bentuk keluaran, sehingga kontrak ONNX dan status GPU langsung terbukti sebelum streaming dimulai.


In [4]:
import threading
from typing import Any

import cv2
import insightface.model_zoo
import numpy as np
import onnxruntime
from insightface import model_zoo
from insightface.app import FaceAnalysis
from insightface.app.common import Face

PROVIDERS = ['CPUExecutionProvider']
if 'CUDAExecutionProvider' in onnxruntime.get_available_providers():
    PROVIDERS = ['CUDAExecutionProvider']
elif 'TensorrtExecutionProvider' in onnxruntime.get_available_providers():
    PROVIDERS = ['TensorrtExecutionProvider']
log('GPU', 'execution providers=' + str(PROVIDERS))

Frame = np.ndarray
THREAD_LOCK = threading.Lock()
FACE_SWAPPER = None
HYPER_SWAPPER = None
PIXEL_BOOST = 256
MASK_PADDING = 2.0
MASK_BLUR = 1.5
MODEL_MEAN = 0.5
MODEL_STANDARD_DEVIATION = 0.5
WARP_TEMPLATE = np.array([
    [0.36167656, 0.40387734],
    [0.63696719, 0.40235469],
    [0.50019687, 0.56044219],
    [0.38710391, 0.72160547],
    [0.61507734, 0.72034453]
], dtype=np.float32)

started = time.time()
FACE_ANALYSER = FaceAnalysis(name='buffalo_l', providers=PROVIDERS)
FACE_ANALYSER.prepare(ctx_id=0, det_size=(640, 640))
log('GPU', 'face analyser siap dalam ' + str(round(time.time() - started, 1)) + 's')


def get_face_swapper() -> Any:
    """Muat model inswapper sekali saja sebagai model cadangan."""
    global FACE_SWAPPER
    with THREAD_LOCK:
        if FACE_SWAPPER is None:
            started_at = time.time()
            FACE_SWAPPER = model_zoo.get_model(MODEL_FILES['inswapper_128'], providers=PROVIDERS)
            log('GPU', 'inswapper siap dalam ' + str(round(time.time() - started_at, 1)) + 's')
    return FACE_SWAPPER


def get_hyperswap_session() -> Any:
    """Muat sesi ONNX HyperSwap sekali saja dan laporkan bentuk input serta outputnya."""
    global HYPER_SWAPPER
    with THREAD_LOCK:
        if HYPER_SWAPPER is None:
            started_at = time.time()
            HYPER_SWAPPER = onnxruntime.InferenceSession(MODEL_PATH, providers=PROVIDERS)
            log('GPU', 'hyperswap siap dalam ' + str(round(time.time() - started_at, 1)) + 's')
            for item in HYPER_SWAPPER.get_inputs():
                log('GPU', 'input ' + item.name + ' shape=' + str(item.shape))
            for item in HYPER_SWAPPER.get_outputs():
                log('GPU', 'output ' + item.name + ' shape=' + str(item.shape))
    return HYPER_SWAPPER


def estimate_affine_matrix(face_landmark_5: Any, crop_size: Any) -> Any:
    """Hitung matriks affine dari 5 titik landmark menuju template arcface_128."""
    normed_template = WARP_TEMPLATE * np.array(crop_size, dtype=np.float32)
    landmarks = np.asarray(face_landmark_5, dtype=np.float32).reshape(5, 2)
    return cv2.estimateAffinePartial2D(landmarks, normed_template, method=cv2.RANSAC, ransacReprojThreshold=100)[0]


def warp_face(temp_frame: Frame, face_landmark_5: Any, crop_size: Any) -> Any:
    """Selaraskan wajah target menjadi crop persegi memakai 5 titik landmark."""
    affine_matrix = estimate_affine_matrix(face_landmark_5, crop_size)
    crop = cv2.warpAffine(temp_frame, affine_matrix, tuple(crop_size), borderMode=cv2.BORDER_REPLICATE, flags=cv2.INTER_AREA)
    return crop, affine_matrix


def normalize_crop(crop: Frame) -> np.ndarray:
    """Ubah crop BGR 0-255 menjadi tensor RGB ternormalisasi rentang -1 sampai 1."""
    normalized = crop[:, :, ::-1].astype(np.float32) / 255.0
    return (normalized - MODEL_MEAN) / MODEL_STANDARD_DEVIATION


def denormalize_crop(tensor: np.ndarray) -> Frame:
    """Ubah tensor keluaran model menjadi crop BGR 0-255."""
    if tensor.ndim == 4 and tensor.shape[1] == 3:
        tensor = tensor[0].transpose(1, 2, 0)
    elif tensor.ndim == 4:
        tensor = tensor[0]
    crop = np.clip(tensor * MODEL_STANDARD_DEVIATION + MODEL_MEAN, 0.0, 1.0)
    return np.ascontiguousarray(crop[:, :, ::-1] * 255).astype(np.uint8)


def static_frame_size(item: Any) -> Any:
    """Ambil ukuran frame statis dari metadata input ONNX, None bila dinamis."""
    dimensions = [dimension for dimension in item.shape if isinstance(dimension, int)]
    if len(dimensions) != 4:
        return None
    if dimensions[1] == 3:
        return dimensions[2], dimensions[3]
    return dimensions[1], dimensions[2]


def channel_first_input(item: Any) -> bool:
    """Tentukan apakah input frame ONNX memakai format NCHW."""
    shape = item.shape
    if len(shape) != 4:
        return True
    if shape[1] == 3:
        return True
    if shape[3] == 3:
        return False
    return True


def prepare_hyperswap_inputs(session: Any, source_embedding: np.ndarray, crop: Frame) -> dict:
    """Susun feed dict sesuai kontrak ONNX HyperSwap, termasuk transposisi channel bila perlu."""
    normalized = normalize_crop(crop)
    inputs = {}
    for item in session.get_inputs():
        if len(item.shape) == 2:
            inputs[item.name] = np.asarray(source_embedding, dtype=np.float32).reshape(1, -1)
            continue
        frame_size = static_frame_size(item)
        resized = normalized
        if frame_size is not None and normalized.shape[:2] != frame_size:
            resized = cv2.resize(normalized, frame_size, interpolation=cv2.INTER_AREA)
        tensor = resized.transpose(2, 0, 1) if channel_first_input(item) else resized
        inputs[item.name] = np.expand_dims(tensor, axis=0).astype(np.float32)
    return inputs


def embedding_dimension(session: Any) -> int:
    """Ambil dimensi embedding statis dari metadata input, default 512."""
    for item in session.get_inputs():
        if len(item.shape) == 2 and isinstance(item.shape[1], int):
            return int(item.shape[1])
    return 512



def create_crop_mask(crop_size: Any) -> np.ndarray:
    """Buat mask kotak bertepi lembut agar sambungan crop tidak terlihat."""
    mask = np.zeros((crop_size[1], crop_size[0]), dtype=np.float32)
    padding = int(round(min(crop_size) * MASK_PADDING / 100))
    mask[padding:crop_size[1] - padding, padding:crop_size[0] - padding] = 1.0
    blur = max(1, int(round(min(crop_size) * MASK_BLUR / 100)))
    return cv2.GaussianBlur(mask, (blur * 2 + 1, blur * 2 + 1), 0)


def paste_back(temp_frame: Frame, crop: Frame, crop_mask: np.ndarray, affine_matrix: Any) -> Frame:
    """Tempelkan crop hasil swap kembali ke frame penuh memakai mask dan matriks kebalikan."""
    inverse_matrix = cv2.invertAffineTransform(affine_matrix)
    frame_size = (temp_frame.shape[1], temp_frame.shape[0])
    inverse_mask = cv2.warpAffine(crop_mask, inverse_matrix, frame_size)[:, :, np.newaxis].clip(0.0, 1.0)
    inverse_crop = cv2.warpAffine(crop, inverse_matrix, frame_size).astype(np.float32)
    blended = inverse_crop * inverse_mask + temp_frame.astype(np.float32) * (1.0 - inverse_mask)
    return blended.clip(0, 255).astype(np.uint8)


def swap_face_hyperswap(source_face: Face, target_face: Face, temp_frame: Frame) -> Frame:
    """Swap satu wajah target memakai HyperSwap 256 lalu tempel kembali ke frame."""
    crop, affine_matrix = warp_face(temp_frame, target_face.kps, (PIXEL_BOOST, PIXEL_BOOST))
    session = get_hyperswap_session()
    inputs = prepare_hyperswap_inputs(session, source_face.normed_embedding, crop)
    output = session.run(None, inputs)[0]
    swapped_crop = denormalize_crop(output)
    if swapped_crop.shape[0] != PIXEL_BOOST or swapped_crop.shape[1] != PIXEL_BOOST:
        swapped_crop = cv2.resize(swapped_crop, (PIXEL_BOOST, PIXEL_BOOST), interpolation=cv2.INTER_AREA)
    return paste_back(temp_frame, swapped_crop, CROP_MASK, affine_matrix)


def swap_face(source_face: Face, target_face: Face, temp_frame: Frame) -> Frame:
    """Tempelkan wajah source ke satu wajah target memakai model yang dipilih di cell model."""
    if SWAP_MODEL == 'inswapper_128':
        return get_face_swapper().get(temp_frame, target_face, source_face, paste_back=True)
    return swap_face_hyperswap(source_face, target_face, temp_frame)


def get_face_analyser() -> Any:
    """Kembalikan analyser wajah lengkap untuk memproses gambar source."""
    return FACE_ANALYSER


def get_one_face(frame: Frame) -> Any:
    """Ambil satu wajah paling kiri, atau None bila tidak ada wajah."""
    faces = get_face_analyser().get(frame)
    try:
        return min(faces, key=lambda face: face.bbox[0])
    except ValueError:
        return None


def get_many_faces(frame: Frame) -> Any:
    """Ambil semua wajah pada frame, atau None bila tidak terdeteksi."""
    try:
        return get_face_analyser().get(frame)
    except IndexError:
        return None


def get_target_faces(frame: Frame) -> Any:
    """
    Deteksi wajah pada frame webcam memakai detektor ringan tanpa model recognition.

    @param {Frame} frame - frame webcam yang akan diproses.
    @returns {Any} daftar wajah tujuan, satu wajah saja bila mode single face.
    """
    faces = FACE_DETECTOR.get(frame)
    if STATE['many_faces']:
        return faces
    try:
        return [min(faces, key=lambda face: face.bbox[0])]
    except ValueError:
        return []


def process_frame(source_face: Face, temp_frame: Frame, manyface: bool) -> Frame:
    """Jalankan face swap sesuai mode single face atau many face."""
    if manyface:
        many_faces = get_many_faces(temp_frame)
        if many_faces:
            for target_face in many_faces:
                temp_frame = swap_face(source_face, target_face, temp_frame)
    else:
        target_face = get_one_face(temp_frame)
        if target_face:
            temp_frame = swap_face(source_face, target_face, temp_frame)
    return temp_frame


CROP_MASK = create_crop_mask((PIXEL_BOOST, PIXEL_BOOST))

DETECT_SIZE = 320
try:
    FACE_DETECTOR = FaceAnalysis(name='buffalo_l', providers=PROVIDERS, allowed_modules=['detection'])
except TypeError:
    FACE_DETECTOR = FaceAnalysis(name='buffalo_l', providers=PROVIDERS)
FACE_DETECTOR.prepare(ctx_id=0, det_size=(DETECT_SIZE, DETECT_SIZE))
log('GPU', 'detektor ringan siap det_size=' + str(DETECT_SIZE) + 'x' + str(DETECT_SIZE))
if PROVIDERS == ['CPUExecutionProvider']:
    log('GPU', 'PERINGATAN: GPU tidak terpakai, inference akan jauh di bawah 1 fps. Runtime > Disconnect and delete runtime, lalu Change runtime type > T4 GPU > Save, jalankan ulang cell instalasi')


def warm_up_swap() -> None:
    """Uji satu inferensi dummy agar kontrak input model dan kesiapan GPU terbukti di awal."""
    if SWAP_MODEL == 'inswapper_128':
        get_face_swapper()
        log('GPU', 'warmup inswapper dilewati, model diuji saat frame pertama')
        return
    session = get_hyperswap_session()
    random_state = np.random.RandomState(0)
    dummy_embedding = random_state.randn(embedding_dimension(session)).astype(np.float32)
    dummy_embedding = dummy_embedding / np.linalg.norm(dummy_embedding)
    dummy_crop = np.full((PIXEL_BOOST, PIXEL_BOOST, 3), 96, dtype=np.uint8)
    try:
        started_at = time.time()
        output = session.run(None, prepare_hyperswap_inputs(session, dummy_embedding, dummy_crop))[0]
        duration_ms = (time.time() - started_at) * 1000
        log('GPU', 'warmup ' + SWAP_MODEL + ' ' + str(round(duration_ms, 1)) + 'ms output=' + str(output.shape)
            + ' rentang=' + str(round(float(output.min()), 2)) + '..' + str(round(float(output.max()), 2)))
    except Exception as error:
        log('GPU', 'warmup gagal ' + type(error).__name__ + ': ' + str(error))


warm_up_swap()
log('GPU', 'model swap aktif=' + SWAP_MODEL + ' pixel_boost=' + str(PIXEL_BOOST))
log('GPU', 'semua model termuat, server siap menerima frame')


02:46:46 [GPU] execution providers=['CUDAExecutionProvider']
02:46:48 [GPU] face analyser siap dalam 2.1s
02:46:50 [GPU] detektor ringan siap det_size=320x320
02:46:51 [GPU] hyperswap siap dalam 0.9s
02:46:51 [GPU] input source shape=[1, 512]
02:46:51 [GPU] input target shape=[1, 3, 256, 256]
02:46:51 [GPU] output output shape=[1, 3, 256, 256]
02:46:51 [GPU] output mask shape=[1, 1, 256, 256]
02:46:53 [GPU] warmup hyperswap_1b_256 1621.6ms output=(1, 3, 256, 256) rentang=-0.33..-0.15
02:46:53 [GPU] model swap aktif=hyperswap_1b_256 pixel_boost=256
02:46:53 [GPU] semua model termuat, server siap menerima frame


## 5. Server Live Streaming

Dua cell di bawah adalah satu kesatuan: jalankan berurutan. Cell pertama berisi penerima source (ZMQ 5555),
decoder webcam (5556), dan encoder hasil (5557). Cell kedua menyalakan worker, watchdog status, dan auto-restart ffmpeg.

Setelah selesai, log akan menampilkan `[BOOT] thread ... jalan` lalu `[SRC] [input] mendengarkan di tcp://127.0.0.1:5555`.


In [5]:
def create_demo_frame() -> Frame:
    """Frame pengganti sebelum GUI mengirim data, agar encoder tetap hidup."""
    frame = np.zeros((STREAM_HEIGHT, STREAM_WIDTH, STREAM_CHANNELS), dtype=np.uint8)
    cv2.putText(frame, 'server ready - waiting for client', (40, STREAM_HEIGHT // 2 - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.putText(frame, 'Out S 5555 | Out T 5556 | In 5557', (40, STREAM_HEIGHT // 2 + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    return frame


def receive_source_bundle(socket_, meta: dict) -> None:
    """Terima metadata, chunk ber-ACK, penanda END, lalu simpan source face."""
    total_chunk = int(meta['total_chunk'])
    log('SRC', 'metadata masuk total_chunk=' + str(total_chunk) + ' size=' + str(meta.get('size')) + ' fps=' + str(meta.get('fps')))
    socket_.setsockopt(zmq.RCVTIMEO, CHUNK_TIMEOUT_MS)
    try:
        socket_.send_string('ACK')
        payload = bytearray()
        for index in range(total_chunk):
            try:
                chunk = socket_.recv()
            except zmq.Again:
                log('SRC', 'chunk ' + str(index + 1) + '/' + str(total_chunk) + ' tidak datang dalam ' + str(round(CHUNK_TIMEOUT_MS / 1000, 1)) + 's')
                raise
            payload.extend(chunk)
            socket_.send_string('ACK ' + str(index + 1) + '/' + str(total_chunk))
            log('SRC', 'chunk ' + str(index + 1) + '/' + str(total_chunk) + ' = ' + str(len(chunk)) + ' byte')
        end_marker = socket_.recv()
        socket_.send_string('Final ACK')
    finally:
        socket_.setsockopt(zmq.RCVTIMEO, META_TIMEOUT_MS)
    if end_marker != b'END':
        log('SRC', 'penanda akhir tak dikenal: ' + repr(end_marker[:16]))
    dtype = np.dtype(meta['dtype_source'])
    shape = tuple(int(value) for value in meta['shape_source'])
    buffer = np.frombuffer(bytes(payload), dtype=dtype)
    if buffer.size != int(np.prod(shape)):
        raise ValueError('payload ' + str(buffer.size) + ' elemen tidak cocok dengan shape ' + str(shape))
    STATE['source_frame'] = buffer.reshape(shape).copy()
    STATE['source_shape'] = shape
    STATE['many_faces'] = bool(meta.get('manyface', False))
    STATE['source_last_at'] = time.time()
    STATE['source_uploads'] += 1
    STATE['source_face'] = None
    STATE['source_face_checked'] = False
    log('SRC', 'upload #' + str(STATE['source_uploads']) + ' siap shape=' + str(shape) + ' dtype=' + str(dtype) + ' manyface=' + str(STATE['many_faces']))


def source_worker() -> None:
    """Layani [input] 5555 terus-menerus untuk banyak sesi Live berurutan."""
    while not STATE['stop'].is_set():
        context = zmq.Context()
        socket_ = context.socket(zmq.REP)
        socket_.setsockopt(zmq.LINGER, 0)
        socket_.setsockopt(zmq.RCVTIMEO, META_TIMEOUT_MS)
        socket_.setsockopt(zmq.SNDTIMEO, 5000)
        try:
            socket_.bind('tcp://127.0.0.1:' + str(PORT_SOURCE))
        except zmq.ZMQError as error:
            log('SRC', 'bind ' + str(PORT_SOURCE) + ' gagal: ' + str(error))
            socket_.close(0)
            context.term()
            time.sleep(1)
            continue
        log('SRC', '[input] mendengarkan di tcp://127.0.0.1:' + str(PORT_SOURCE))
        try:
            while not STATE['stop'].is_set():
                try:
                    meta = socket_.recv_json()
                except zmq.Again:
                    continue
                receive_source_bundle(socket_, meta)
        except Exception as error:
            STATE['errors'] += 1
            log('SRC', 'socket direset setelah ' + type(error).__name__ + ': ' + str(error))
            traceback.print_exc()
        finally:
            socket_.close(0)
            context.term()
        time.sleep(0.3)


def temp_worker() -> None:
    """Dekode stream webcam dari GUI di [inputtwo] 5556 menjadi frame BGR."""
    frame_bytes = STREAM_WIDTH * STREAM_HEIGHT * STREAM_CHANNELS
    while not STATE['stop'].is_set():
        command = [
            'ffmpeg', '-hide_banner', '-loglevel', 'warning',
            '-flags', 'low_delay',
            '-f', 'mpegts', '-i', 'tcp://127.0.0.1:' + str(PORT_TEMP) + '?listen=1&fifo_size=100000&overrun_nonfatal=1',
            '-f', 'rawvideo', '-pix_fmt', 'bgr24',
            '-s', str(STREAM_WIDTH) + 'x' + str(STREAM_HEIGHT), 'pipe:1'
        ]
        log('TEM', 'decoder start di tcp://127.0.0.1:' + str(PORT_TEMP))
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        started_at = time.time()
        try:
            while not STATE['stop'].is_set():
                raw = process.stdout.read(frame_bytes)
                if not raw or len(raw) < frame_bytes:
                    log('TEM', 'decoder berhenti setelah ' + str(round(time.time() - started_at, 1)) + 's')
                    break
                frame = np.frombuffer(raw, dtype=np.uint8).reshape((STREAM_HEIGHT, STREAM_WIDTH, STREAM_CHANNELS))
                TEMP_BUFFER.append(frame.astype(np.uint8, copy=True))
                STATE['temp_frames'] += 1
                STATE['temp_last_at'] = time.time()
        except Exception as error:
            STATE['errors'] += 1
            log('TEM', 'decoder error ' + type(error).__name__ + ': ' + str(error))
        finally:
            process.terminate()
            try:
                process.wait(timeout=3)
            except Exception:
                process.kill()
            tail = process.stderr.read().decode('utf-8', 'ignore').strip().splitlines()[-4:]
            for line in tail:
                log('TEM', 'ffmpeg: ' + line)
        if not STATE['stop'].is_set():
            time.sleep(1)


In [6]:
def resolve_source_face() -> Any:
    """Ubah gambar source hasil upload menjadi objek Face sekali saja per upload."""
    if STATE['source_face'] is not None:
        return STATE['source_face']
    if STATE['source_frame'] is None or STATE['source_face_checked']:
        return None
    STATE['source_face_checked'] = True
    face = get_one_face(STATE['source_frame'])
    if face is None:
        log('SWAP', 'wajah tidak terdeteksi pada source: upload ulang foto yang wajahnya jelas dan menghadap kamera')
        return None
    STATE['source_face'] = face
    log('SWAP', 'source wajah siap bbox=' + str([round(float(value), 1) for value in face.bbox]))
    return face


SWAP_STATS = {'frames': 0, 'detect_s': 0.0, 'swap_s': 0.0}


def log_swap_stats() -> None:
    """Catat rata-rata waktu deteksi dan swap supaya plafon fps terbaca dari log."""
    if SWAP_STATS['frames'] % 5 != 0:
        return
    count = SWAP_STATS['frames']
    detect_ms = SWAP_STATS['detect_s'] / count * 1000
    swap_ms = SWAP_STATS['swap_s'] / count * 1000
    log('OUT', SWAP_MODEL + ' inferensi ' + str(round(detect_ms, 1)) + 'ms deteksi + ' + str(round(swap_ms, 1))
        + 'ms swap = ' + str(round(detect_ms + swap_ms, 1)) + 'ms/frame')


def apply_swap(frame: Frame) -> Frame:
    """Swap frame webcam dengan source aktif, kirim frame asli saat inference gagal."""
    source_face = resolve_source_face()
    if source_face is None:
        return frame
    started_at = time.time()
    try:
        targets = get_target_faces(frame)
        detected_at = time.time()
        for target_face in targets:
            frame = swap_face(source_face, target_face, frame)
        finished_at = time.time()
        SWAP_STATS['frames'] += 1
        SWAP_STATS['detect_s'] += detected_at - started_at
        SWAP_STATS['swap_s'] += finished_at - detected_at
        log_swap_stats()
        return frame
    except Exception as error:
        STATE['errors'] += 1
        log('SWAP', 'error inference ' + type(error).__name__ + ': ' + str(error))
        return frame


def output_worker() -> None:
    """Encode frame hasil ke mpegts di [output] 5557 untuk dibaca GUI."""
    demo_frame = create_demo_frame()
    while not STATE['stop'].is_set():
        command = [
            'ffmpeg', '-hide_banner', '-loglevel', 'warning',
            '-fflags', 'nobuffer', '-flags', 'low_delay', '-probesize', '32',
            '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', str(STREAM_WIDTH) + 'x' + str(STREAM_HEIGHT),
            '-use_wallclock_as_timestamps', '1', '-i', 'pipe:',
            '-c:v', 'libx264', '-preset', 'ultrafast', '-tune', 'zerolatency',
            '-g', GOP, '-b:v', BITRATE, '-maxrate', MAXRATE, '-bufsize', BUFSIZE, '-threads', '2',
            '-x264-params', 'repeat-headers=1', '-fps_mode', 'passthrough',
            '-mpegts_flags', '+resend_headers', '-flush_packets', '1',
            '-f', 'mpegts', 'tcp://127.0.0.1:' + str(PORT_OUT) + '?listen=1&fifo_size=50000&overrun_nonfatal=1'
        ]
        log('OUT', 'encoder start di tcp://127.0.0.1:' + str(PORT_OUT))
        process = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
        started_at = time.time()
        try:
            while not STATE['stop'].is_set():
                if TEMP_BUFFER:
                    result = apply_swap(TEMP_BUFFER[-1].copy())
                else:
                    result = demo_frame
                if result.shape[0] != STREAM_HEIGHT or result.shape[1] != STREAM_WIDTH:
                    result = cv2.resize(result, (STREAM_WIDTH, STREAM_HEIGHT))
                process.stdin.write(np.ascontiguousarray(result).tobytes())
                process.stdin.flush()
                STATE['out_frames'] += 1
                STATE['out_last_at'] = time.time()
                time.sleep(1 / FRAME_RATE)
        except (BrokenPipeError, OSError) as error:
            log('OUT', 'pipe encoder tertutup setelah ' + str(round(time.time() - started_at, 1)) + 's: ' + str(error))
        except Exception as error:
            STATE['errors'] += 1
            log('OUT', 'encoder error ' + type(error).__name__ + ': ' + str(error))
        finally:
            try:
                process.stdin.close()
            except Exception:
                pass
            process.terminate()
            try:
                process.wait(timeout=3)
            except Exception:
                process.kill()
            tail = process.stderr.read().decode('utf-8', 'ignore').strip().splitlines()[-4:]
            for line in tail:
                log('OUT', 'ffmpeg: ' + line)
        if not STATE['stop'].is_set():
            time.sleep(1)


def watchdog_worker() -> None:
    """Cetak status tiap beberapa detik beserta saran perbaikan otomatis."""
    while not STATE['stop'].is_set():
        time.sleep(STATUS_INTERVAL_S)
        log('STAT', status_line())
        for message in hints():
            log('HINT', message)


def start_server() -> None:
    """Bebaskan port lalu jalankan semua worker Live Streaming."""
    if any(thread.is_alive() for thread in STATE['threads'].values()):
        log('BOOT', 'server sudah berjalan, tidak start ulang')
        return
    STATE['stop'].clear()
    kill_port_users([PORT_SOURCE, PORT_TEMP, PORT_OUT])
    workers = {'source': source_worker, 'temp': temp_worker, 'output': output_worker, 'watchdog': watchdog_worker}
    for name, target in workers.items():
        thread = threading.Thread(target=target, name='dlc-' + name, daemon=True)
        thread.start()
        STATE['threads'][name] = thread
        log('BOOT', 'thread ' + name + ' jalan')
    time.sleep(3)
    log('STAT', status_line())
    for message in hints():
        log('HINT', message)


def stop_server() -> None:
    """Hentikan semua worker dan lepaskan port 5555 sampai 5557."""
    STATE['stop'].set()
    kill_port_users([PORT_SOURCE, PORT_TEMP, PORT_OUT])
    for name, thread in list(STATE['threads'].items()):
        thread.join(timeout=3)
        log('BOOT', 'thread ' + name + ' dihentikan')
    STATE['threads'].clear()


start_server()


02:46:53 [SRC] [input] mendengarkan di tcp://127.0.0.1:5555
02:46:53 [BOOT] thread source jalan
02:46:53 [TEM] decoder start di tcp://127.0.0.1:5556
02:46:53 [BOOT] thread temp jalan
02:46:53 [OUT] encoder start di tcp://127.0.0.1:5557
02:46:53 [BOOT] thread output jalan
02:46:53 [BOOT] thread watchdog jalan
02:46:56 [STAT] source_uploads=0 source_ready=False source_idle=None temp_frames=0 temp_idle=None out_frames=1 out_idle=2.9 errors=0 buffer=0
02:46:56 [HINT] belum ada upload source: cek GUI Out S -> [input] 5555 lalu klik Live


## 6. Tunnel ngrok (3 port sekaligus)

Isi `AUTHTOKEN` dengan authtoken ngrok Anda (dashboard -> Your Authtoken), jalankan cell,
lalu **copy tiga URL** yang dicetak ke kolom GUI sesuai urutan: `In:` = output, `Out S:` = input, `Out T:` = inputtwo.

Jangan pernah mencampur URL dari run yang berbeda: kalau cell ini dijalankan ulang, ganti ketiga kolom GUI.


In [7]:
!wget -q -nc https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz
!chmod +x ngrok

AUTHTOKEN = '37kqTtQS0u9equrpXPTSdcUg4Nd_87vTAkHrHQ2THLFKpCtQF'
REGION = 'ap'


def free_tunnels() -> None:
    """Hentikan proses ngrok lama agar config baru dipakai."""
    subprocess.run('pkill -f ngrok', shell=True, capture_output=True)
    time.sleep(1)
    log('TUN', 'proses ngrok lama dihentikan')


def write_config(authtoken: str, region: str) -> str:
    """Tulis file config ngrok berisi tiga tunnel TCP."""
    text = (
        'version: "2"\n'
        'authtoken: ' + authtoken + '\n'
        'region: ' + region + '\n'
        'console_ui: false\n'
        'log: /content/ngrok_live.log\n'
        'tunnels:\n'
        '  input:\n'
        '    addr: ' + str(PORT_SOURCE) + '\n'
        '    proto: tcp\n'
        '  inputtwo:\n'
        '    addr: ' + str(PORT_TEMP) + '\n'
        '    proto: tcp\n'
        '  output:\n'
        '    addr: ' + str(PORT_OUT) + '\n'
        '    proto: tcp\n'
    )
    path = '/content/ngrok_live.yml'
    with open(path, 'w', encoding='utf-8') as handle:
        handle.write(text)
    log('TUN', 'config ditulis di ' + path)
    return path


def print_mapping(tunnels: dict) -> None:
    """Cetak pemetaan siap copy ke kolom In, Out S, Out T di GUI."""
    fields = [('In:', 'output'), ('Out S:', 'input'), ('Out T:', 'inputtwo')]
    print('=' * 78)
    for label, name in fields:
        print(label.ljust(8) + str(tunnels.get(name, 'TIDAK ADA')))
    print('=' * 78)
    for label, name in fields:
        if name not in tunnels:
            log('TUN', 'tunnel ' + name + ' tidak terbuka, kolom GUI ' + label + ' akan gagal')
    if any(name not in tunnels for _, name in fields):
        log('TUN', 'jalankan ulang cell tunnel; kalau tetap kurang, buka ' + NGROK_LOG_PATH)


EXPECTED_TUNNELS = ('input', 'inputtwo', 'output')
NGROK_LOG_PATH = '/content/ngrok_live.log'


def ngrok_log_tail(lines: int = 12) -> str:
    """Ambil baris terakhir log ngrok sebagai alasan bila tunnel gagal dibuka."""
    if not os.path.exists(NGROK_LOG_PATH):
        return 'log ngrok belum ada'
    with open(NGROK_LOG_PATH, 'r', encoding='utf-8', errors='replace') as handle:
        rows = handle.readlines()
    return ' | '.join(row.strip() for row in rows[-lines:] if row.strip())


def fetch_tunnels(attempts: int = 20) -> dict:
    """Tunggu API lokal ngrok sampai tiga tunnel utama muncul atau waktu habis."""
    import urllib.request
    found = {}
    for _ in range(attempts):
        try:
            with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
                payload = json.loads(response.read().decode('utf-8'))
            found = {tunnel['name']: tunnel['public_url'] for tunnel in payload['tunnels']}
            if all(name in found for name in EXPECTED_TUNNELS):
                return found
        except Exception:
            pass
        time.sleep(1)
    return found


def open_tunnels() -> dict:
    """Jalankan ngrok untuk tiga tunnel dan laporkan kegagalan beserta outputnya."""
    if not AUTHTOKEN:
        log('TUN', 'AUTHTOKEN kosong: isi authtoken ngrok di variabel AUTHTOKEN lalu jalankan ulang cell')
        return {}
    config_path = write_config(AUTHTOKEN, REGION)
    free_tunnels()
    process = subprocess.Popen(
        ['./ngrok', 'start', '--all', '--config', config_path],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    tunnels = fetch_tunnels()
    missing = [name for name in EXPECTED_TUNNELS if name not in tunnels]
    if missing:
        log('TUN', 'tunnel belum muncul: ' + ','.join(missing) + ' | log ngrok: ' + ngrok_log_tail())
    if not tunnels:
        process.terminate()
        try:
            output = process.stdout.read()[-1500:]
        except Exception:
            output = 'output tidak terbaca'
        log('TUN', 'gagal membuka tunnel: ' + output.replace('\n', ' | '))
        return {}
    STATE['tunnels'] = tunnels
    for name, url in tunnels.items():
        log('TUN', name + ' -> ' + url)
    print_mapping(tunnels)
    return tunnels


open_tunnels()


02:46:57 [TUN] config ditulis di /content/ngrok_live.yml
02:46:58 [STAT] source_uploads=0 source_ready=False source_idle=None temp_frames=0 temp_idle=None out_frames=1 out_idle=4.9 errors=0 buffer=0
02:46:58 [HINT] belum ada upload source: cek GUI Out S -> [input] 5555 lalu klik Live
02:46:58 [TUN] proses ngrok lama dihentikan
02:46:59 [TUN] input -> tcp://0.tcp.ap.ngrok.io:10462
02:46:59 [TUN] inputtwo -> tcp://0.tcp.ap.ngrok.io:22611
02:46:59 [TUN] output -> tcp://0.tcp.ap.ngrok.io:28524
In:     tcp://0.tcp.ap.ngrok.io:28524
Out S:  tcp://0.tcp.ap.ngrok.io:10462
Out T:  tcp://0.tcp.ap.ngrok.io:22611


{'input': 'tcp://0.tcp.ap.ngrok.io:10462',
 'inputtwo': 'tcp://0.tcp.ap.ngrok.io:22611',
 'output': 'tcp://0.tcp.ap.ngrok.io:28524'}

## 7. Auto-diagnosa

Cell ini melakukan semua pemeriksaan sekaligus dan mencetak penyebab bila ada yang salah:

1. versi library + GPU, 2. pemakai port 5555/5556/5557, 3. status thread server,
4. keterjangkauan tiga URL ngrok, 5. **self-test ZMQ** ke `Out S:` lokal (meniru upload source dari GUI),
6. snapshot status setelah 5 detik + saran perbaikan.

Kalau `self test ZMQ 5555: LULUS` berarti jalur `Out S:` di sisi server sehat, jadi masalah ada di URL tunnel.


## 8. Lihat log lengkap

Untuk memantau saat GUI sudah terhubung, jalankan cell ini berulang kali.
Untuk menghentikan server, jalankan `stop_server()` pada cell baru.


In [8]:
!tail -n 80 /content/live_debug.log


02:44:15 [HINT] belum ada upload source: cek GUI Out S -> [input] 5555 lalu klik Live
02:44:15 [SRC] chunk 6/11 = 249429 byte
02:44:16 [SRC] chunk 7/11 = 249429 byte
02:44:16 [SRC] chunk 8/11 = 249429 byte
02:44:17 [SRC] chunk 9/11 = 249429 byte
02:44:17 [SRC] chunk 10/11 = 249429 byte
02:44:17 [SRC] chunk 11/11 = 6 byte
02:44:17 [SRC] upload #1 siap shape=(1029, 808, 3) dtype=uint8 manyface=False
02:44:20 [STAT] source_uploads=1 source_ready=True source_idle=2.4 temp_frames=0 temp_idle=None out_frames=90 out_idle=0.0 errors=0 buffer=0
02:44:25 [STAT] source_uploads=1 source_ready=True source_idle=7.4 temp_frames=60 temp_idle=0.0 out_frames=324 out_idle=0.8 errors=0 buffer=4
02:44:25 [SWAP] wajah tidak terdeteksi pada source: upload ulang foto yang wajahnya jelas dan menghadap kamera
02:44:30 [STAT] source_uploads=1 source_ready=True source_idle=12.4 temp_frames=323 temp_idle=0.0 out_frames=583 out_idle=0.0 errors=0 buffer=4
02:44:35 [STAT] source_uploads=1 source_ready=True source_idl